In [32]:
%pip install -q -U "crewai[google-genai]" ddgs langgraph mlflow ragas pandas

In [39]:
import os
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API key:AQ.Ab8RN6KJMeh4N11whumjOf_vc83IW4NYKsxt-SZHS4xiockWFA ").strip()

if not GEMINI_API_KEY:
    raise ValueError("API key cannot be empty.")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ.pop("GOOGLE_API_KEY", None)

print("Gemini API key loaded.")

Enter your Gemini API key:AQ.Ab8RN6KJMeh4N11whumjOf_vc83IW4NYKsxt-SZHS4xiockWFA ··········
Gemini API key loaded.


In [40]:
from google import genai
from google.genai import types

client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(timeout=30000)
)

try:
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents="Reply with exactly: GEMINI API WORKS"
    )
    print(response.text)
except Exception as e:
    print("Gemini test failed:", type(e).__name__)
    print(str(e)[:1000])

GEMINI API WORKS


In [42]:
from crewai import LLM

gemini_llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    temperature=0.2
)

try:
    test_result = gemini_llm.call(
        "Reply with exactly: CREWAI GEMINI WORKS"
    )
    print(test_result)
except Exception as e:
    print("CrewAI Gemini test failed:", str(e)[:800])

CREWAI GEMINI WORKS


In [43]:
from ddgs import DDGS

topic = "Generative AI in Education"

search_results = DDGS().text(
    "Generative AI in education benefits risks research",
    max_results=5
)

research_notes = "\n\n".join(
    f"Title: {item.get('title', '')}\n"
    f"URL: {item.get('href', '')}\n"
    f"Summary: {item.get('body', '')}"
    for item in search_results
)

print("Research sources found:", len(search_results))
print(research_notes)

Research sources found: 5
Title: Generative Artificial Intelligence in Education: Risks and ...
URL: https://link.springer.com/chapter/10.1007/978-3-031-72549-4_11
Summary: Nov 1, 2024 · This study explores the possibilities and challenges of artificial intelligence in higher education, specifically the role of Generative Artificial Intelligence in the classroom in order to achieve a more personalized and inclusive university education that enhances...

Title: Guidance for generative AI in education and research - UNESCO
URL: https://www.unesco.org/en/articles/guidance-generative-ai-education-and-research
Summary: Sep 7, 2023 · To guide the proper use of the tools in education and research, this Guidance proposes a human-agent and age-appropriate approach to the ethical validation and pedagogical design processes.

Title: Generation AI: Navigating the opportunities and risks of ...
URL: https://www.unesco.org/en/articles/generation-ai-navigating-opportunities-and-risks-artificial-intel

In [44]:
researcher = Agent(
    role="Research Analyst",
    goal=(
        "Analyze research about Generative AI in Education, "
        "identify findings, benefits, risks, and preserve source URLs."
    ),
    backstory=(
        "You are a careful research analyst who distinguishes "
        "evidence from assumptions and never invents sources."
    ),
    llm=gemini_llm,
    verbose=False
)

print("Researcher agent created.")

Researcher agent created.


In [45]:
writer = Agent(
    role="Research Report Writer",
    goal="Write a structured, clear, balanced research report.",
    backstory="You turn research findings into readable reports.",
    llm=gemini_llm,
    verbose=False
)

reviewer = Agent(
    role="Quality Reviewer",
    goal="Review the report for accuracy, clarity, balance, and completeness.",
    backstory="You flag unsupported claims and improve research reports.",
    llm=gemini_llm,
    verbose=False
)

print("Writer agent created.")
print("Reviewer agent created.")
print("All three agents are ready.")

Writer agent created.
Reviewer agent created.
All three agents are ready.


In [46]:
from crewai import Task

research_task = Task(
    description=(
        "Research {topic} using the supplied research notes. "
        "Identify key findings, benefits, risks, examples, and limitations. "
        "Preserve source URLs and do not invent evidence.\n\n"
        "Research notes:\n{research_notes}"
    ),
    expected_output="Organized findings with source URLs, benefits, and risks.",
    agent=researcher
)

print("Research task created.")

Research task created.


In [47]:
writing_task = Task(
    description=(
        "Write a structured report about {topic} using the research findings. "
        "Include an introduction, key findings, benefits, risks, "
        "recommendations, limitations, conclusion, and source URLs."
    ),
    expected_output="A complete, balanced research report with headings.",
    agent=writer,
    context=[research_task]
)

print("Writing task created.")

Writing task created.


In [49]:
review_task = Task(
    description=(
        "Review the draft report for clarity, balance, completeness, "
        "and unsupported claims. Preserve source URLs and produce "
        "the final improved report."
    ),
    expected_output="A reviewed, improved final research report.",
    agent=reviewer,
    context=[writing_task]
)

print("Review task created.")

Review task created.


In [50]:
from crewai import Crew, Process

research_crew = Crew(
    agents=[researcher, writer, reviewer],
    tasks=[research_task, writing_task, review_task],
    process=Process.sequential,
    verbose=False
)

print("CrewAI pipeline created.")

CrewAI pipeline created.


In [51]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class ResearchState(TypedDict):
    topic: str
    research_notes: str
    report: str
    status: str

print("LangGraph state defined.")

LangGraph state defined.


In [52]:
async def run_research(state: ResearchState):
    result = await research_crew.kickoff_async(
        inputs={
            "topic": state["topic"],
            "research_notes": state["research_notes"]
        }
    )
    return {
        "report": str(result),
        "status": "completed"
    }

builder = StateGraph(ResearchState)
builder.add_node("research_pipeline", run_research)
builder.add_edge(START, "research_pipeline")
builder.add_edge("research_pipeline", END)

research_graph = builder.compile()

print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


In [53]:
result = await research_graph.ainvoke({
    "topic": topic,
    "research_notes": research_notes,
    "report": "",
    "status": "started"
})

final_report = result["report"]

print("Workflow status:", result["status"])
print("\nFINAL RESEARCH REPORT")
print("=" * 60)
print(final_report)

Workflow status: completed

FINAL RESEARCH REPORT
# Research Report: Generative AI in Education

## 1. Introduction
Generative Artificial Intelligence (GenAI)—primarily encompassing large language model tools capable of generating text and code based on user prompts—is reshaping the educational landscape. As these technologies proliferate across classrooms and lecture halls, educators, researchers, and policymakers are tasked with understanding how to integrate them effectively. This report examines the current state of GenAI in education, highlighting key findings, potential benefits, associated risks and limitations, and actionable recommendations to guide its future use.

## 2. Key Findings
* **Rapid Integration in Higher Education:** GenAI is quickly becoming a standard part of everyday practice in higher education settings, moving from an experimental novelty to an integrated digital tool ([ScienceDirect Review](https://www.sciencedirect.com/science/article/pii/S2666374026000130))

In [54]:
from pathlib import Path

report_path = Path("W9D5_Generative_AI_Education_Report.md")
report_path.write_text(final_report, encoding="utf-8")

print("Report saved:", report_path)
print("Report size:", report_path.stat().st_size, "bytes")

Report saved: W9D5_Generative_AI_Education_Report.md
Report size: 6176 bytes


In [55]:
import mlflow
import time

mlflow.set_experiment("W9D5_Automated_Research_Report")

with mlflow.start_run(run_name="Gemini_CrewAI_Research"):
    mlflow.log_param("topic", topic)
    mlflow.log_param("model", "gemini-3.5-flash-lite")
    mlflow.log_param("framework", "CrewAI + LangGraph")
    mlflow.log_metric("report_characters", len(final_report))
    mlflow.log_metric("research_sources", len(search_results))

    mlflow.log_artifact(str(report_path))

print("MLflow run logged successfully.")

2026/10/12 13:41:17 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/12 13:41:17 INFO mlflow.store.db.utils: Updating database tables
2026/10/12 13:41:21 INFO mlflow.tracking.fluent: Experiment with name 'W9D5_Automated_Research_Report' does not exist. Creating a new experiment.


MLflow run logged successfully.


In [56]:
import pandas as pd

evaluation_data = pd.DataFrame([
    {
        "question": "What are the main benefits of Generative AI in education?",
        "answer": final_report[:3000],
        "contexts": [research_notes],
        "ground_truth": (
            "Potential benefits include personalized learning, "
            "learning support, feedback, and assistance for educators. "
            "Benefits depend on implementation and evidence."
        )
    },
    {
        "question": "What are the main risks of Generative AI in education?",
        "answer": final_report[:3000],
        "contexts": [research_notes],
        "ground_truth": (
            "Potential risks include inaccurate outputs, bias, privacy concerns, "
            "overreliance, academic integrity issues, and unequal access."
        )
    }
])

evaluation_data.to_csv(
    "W9D5_ragas_evaluation_dataset.csv",
    index=False
)

print(evaluation_data[["question", "answer"]])
print("\nEvaluation dataset saved.")

                                            question  \
0  What are the main benefits of Generative AI in...   
1  What are the main risks of Generative AI in ed...   

                                              answer  
0  # Research Report: Generative AI in Education\...  
1  # Research Report: Generative AI in Education\...  

Evaluation dataset saved.


In [57]:
required_sections = [
    "introduction",
    "benefits",
    "risks",
    "recommendations",
    "conclusion"
]

report_lower = final_report.lower()

quality_results = {
    section: section in report_lower
    for section in required_sections
}

for section, found in quality_results.items():
    print(f"{section.title()}: {'Found' if found else 'Not found'}")

print("\nReport word count:", len(final_report.split()))
print("Report has content:", bool(final_report.strip()))

Introduction: Found
Benefits: Found
Risks: Found
Recommendations: Found
Conclusion: Found

Report word count: 634
Report has content: True


In [58]:
from pathlib import Path

submission_files = [
    "W9D5_Generative_AI_Education_Report.md",
    "W9D5_ragas_evaluation_dataset.csv"
]

for filename in submission_files:
    path = Path(filename)
    print(f"{filename}: {'Ready' if path.exists() else 'Missing'}")

W9D5_Generative_AI_Education_Report.md: Ready
W9D5_ragas_evaluation_dataset.csv: Ready


In [59]:
from google.colab import files

files.download("W9D5_Generative_AI_Education_Report.md")
files.download("W9D5_ragas_evaluation_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>